# OpenAI Realtime + LangSmith

This notebook keeps the workshop-facing pieces visible: the Realtime session shape and the LangSmith tracing setup. The final run cell uses the maintained backend implementation through `workshop`.

## 1. Build The Agent

The speech-to-speech setup is the Realtime session configuration: instructions, audio settings, and the weather tool schema the model can call mid-conversation.

In [ ]:
import os
import uuid

from dotenv import load_dotenv

from workshop import ConsoleStatus, MicStream, SpeakerStream, run_openai_realtime

load_dotenv()

PROJECT = "voice-workshop-openai-realtime"
MODEL = os.getenv("REALTIME_MODEL", "gpt-realtime-2")
SAMPLE_RATE = 24_000

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
SYSTEM_PROMPT = """You are a friendly voice assistant who can look up the
weather for any city. Keep replies short, conversational, and free of
formatting. When the user asks about weather, call lookup_weather once per
city, then summarize the result in one or two spoken sentences."""

WEATHER_TOOL = {
    "type": "function",
    "name": "lookup_weather",
    "description": "Get the current weather for a single city. Call once per city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Paris."}
        },
        "required": ["city"],
    },
}

SESSION = {
    "type": "realtime",
    "instructions": SYSTEM_PROMPT,
    "output_modalities": ["audio"],
    "audio": {
        "input": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "transcription": {"model": "gpt-4o-mini-transcribe"},
            "noise_reduction": {"type": "near_field"},
            "turn_detection": {
                "type": "server_vad",
                "create_response": False,
                "interrupt_response": True,
            },
        },
        "output": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "voice": "alloy",
        },
    },
    "tools": [WEATHER_TOOL],
    "tool_choice": "auto",
}

## 2. Configure Tracing

The LangSmith integration wraps the raw OpenAI Realtime connection. The wrapper observes the event stream and records the user and agent audio timelines.

In [ ]:
from langsmith.integrations.openai_realtime import wrap_realtime

thread_id = str(uuid.uuid4())


def trace_realtime(raw_connection, *, is_agent_speaking):
    return wrap_realtime(
        raw_connection,
        thread_id=thread_id,
        sample_rate=SAMPLE_RATE,
        project_name=PROJECT,
        tags=["workshop", "openai-realtime"],
        metadata={"model": MODEL},
        is_agent_speaking=is_agent_speaking,
    )


thread_id

## 3. Put It Together

The app opens a Realtime connection, applies the session config, wraps it for tracing, then streams audio and tool results through that traced connection.

In [ ]:
from contextlib import asynccontextmanager

from openai import AsyncOpenAI

client = AsyncOpenAI()


@asynccontextmanager
async def traced_agent_connection(audio_out):
    async with client.realtime.connect(model=MODEL) as raw_connection, trace_realtime(
        raw_connection,
        is_agent_speaking=lambda: audio_out.buffered_bytes() > 0,
    ) as connection:
        await connection.session.update(session=SESSION)
        audio_out.set_played_callback(connection.record_agent_audio)
        yield connection


## 4. Run The Agent Live

The cells above show the shape of the implementation. This cell uses the shared backend code so the workshop does not maintain a second Realtime event loop.

In [ ]:
audio_in = MicStream(sample_rate=SAMPLE_RATE)
audio_out = SpeakerStream(sample_rate=SAMPLE_RATE)
ui = ConsoleStatus()

await run_openai_realtime(PROJECT, audio_in=audio_in, audio_out=audio_out, ui=ui)